## NA2M — single pass through Stage 1 + FAST screening

In [23]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

### Load config

In [24]:
from na2m.utils.config import load_na2m_config

CONFIG_PATH = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_tuned.yaml'

config = load_na2m_config(CONFIG_PATH)
config.top_m = 10  # NA2M-specific, not in tuned yaml

print(config)

NA2MConfig(dataset_path='C:\\Users\\maart\\OneDrive\\Documenten\\Universiteit\\Scriptie\\python_repo\\thesis-nam\\datasets\\raw\\compas-scores-two-years.csv', num_units=64, hidden_sizes=[64, 32], activation='relu', dropout=0.1, inter_units=32, inter_hidden=[], feature_dropout=0.05, output_regularization=0.03242773404285635, l2_regularization=3.002585238683321e-05, clarity_regularization=0.0, lr=0.027365104862115727, decay_rate=0.995, val_frac=0.15, test_frac=0.15, seed=42, pool_val_frac=0.15, batch_size=1024, num_epochs=100, patience=50, val_check_interval=10, top_m=10, eta_prune=0.0, block_train_epochs=100, finetune_epochs=100, concurvity_filter=True, concurvity_threshold=0.5, k_folds=5, fold_seed=42, seeds=[0, 1, 2, 3, 4], grid_size=100, task='classification')


### Load and preprocess data

In [25]:
from na2m.data.data_utils import load_compas, preprocess, split

DATA_PATH = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv'

df = load_compas(DATA_PATH)
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}', end='')
    if fm.type == 'cat':
        print(f'  n_levels={fm.n_levels}')
    else:
        print(f'  [{fm.min:.3f}, {fm.max:.3f}]')

Samples: 6907, Features: 6
  age                       type=num  [18.000, 96.000]
  priors_count              type=num  [0.000, 38.000]
  length_of_stay            type=num  [-1.000, 799.000]
  c_charge_degree           type=cat  n_levels=2
  race                      type=cat  n_levels=6
  sex                       type=cat  n_levels=2


### Split and build DataLoaders

In [26]:
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, val_frac=0.15, test_frac=0.15, seed=SEED
)

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val,   y_val)
pool_dataset  = NAMDataset(
    np.concatenate([X_train, X_val]),
    np.concatenate([y_train, y_val]),
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(pool_dataset,  batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Pool: {len(pool_dataset)}')

Train: 4834, Val: 1036, Pool: 5870


### Build the NA2M model

In [27]:
from na2m.models.na2m import NA2M

set_seed(SEED)

model = NA2M(
    num_features=X.shape[1],
    feature_meta=feature_meta,
    num_units=config.num_units,
    hidden_sizes=config.hidden_sizes,
    dropout=config.dropout,
    feature_dropout=config.feature_dropout,
    activation=config.activation,
    inter_units=config.inter_units,
    inter_hidden=config.inter_hidden,
)
print(model)

NA2M(
  (main_nns): ModuleList(
    (0-2): 3 x FeatureNN(
      (dropout): Dropout(p=0.1, inplace=False)
      (model): Sequential(
        (0): LinReLU()
        (1): Dropout(p=0.1, inplace=False)
        (2): Linear(in_features=64, out_features=64, bias=True)
        (3): ReLU()
        (4): Dropout(p=0.1, inplace=False)
        (5): Linear(in_features=64, out_features=32, bias=True)
        (6): ReLU()
        (7): Dropout(p=0.1, inplace=False)
        (8): Linear(in_features=32, out_features=1, bias=False)
      )
    )
    (3): CategNet(
      (output_layer): Linear(in_features=2, out_features=1, bias=False)
    )
    (4): CategNet(
      (output_layer): Linear(in_features=6, out_features=1, bias=False)
    )
    (5): CategNet(
      (output_layer): Linear(in_features=2, out_features=1, bias=False)
    )
  )
  (inter_nns): ModuleDict()
  (dropout_layer): Dropout(p=0.05, inplace=False)
)


### Stage 1 — train main effects

In [28]:
from na2m.training.fit_na2m import stage1_main

stage1_main(model, train_loader, val_loader, pool_loader, config)
print('Stage 1 done — main effects trained and centered.')

Epoch 1/100| Epoch loss = 0.7256 | best=0.0000
Epoch 100/100| Epoch loss = 0.6192 | best=0.7332
Stage 1 done — main effects trained and centered.


### Stage 1 prediction score (AUROC on test set)

In [29]:
from nam.training.metrics import auroc

model.eval()
all_logits, all_targets = [], []

test_dataset_nb = NAMDataset(X_test, y_test)
test_loader_nb  = DataLoader(test_dataset_nb, batch_size=config.batch_size, shuffle=False)

with torch.no_grad():
    for X_batch, y_batch, _ in test_loader_nb:
        logits, _ = model(X_batch)
        all_logits.append(logits)
        all_targets.append(y_batch)

test_auc = auroc(torch.cat(all_logits), torch.cat(all_targets))
print(f'Stage 1 test AUROC: {test_auc:.4f}')

Stage 1 test AUROC: 0.7549


### FAST screening — rank all candidate pairs

In [30]:
from na2m.selection.fast import verify_fast_available, fast_screen

print('FAST available:', verify_fast_available())

# Collect the full pool as numpy arrays for the screener
X_pool = np.concatenate([X_train, X_val])
y_pool = np.concatenate([y_train, y_val])

ranked_pairs = fast_screen(model, X_pool, y_pool, task=config.task)

print(f'\nAll {len(ranked_pairs)} pairs ranked by FAST interaction strength (best first):')
for rank, (j, k) in enumerate(ranked_pairs):
    print(f'  {rank+1:2d}. ({feature_meta[j].name}, {feature_meta[k].name})')

top_m = ranked_pairs[:config.top_m]
print(f'\nTop-{config.top_m} selected for block training:')
for j, k in top_m:
    print(f'  ({feature_meta[j].name}, {feature_meta[k].name})')

FAST available: True

All 15 pairs ranked by FAST interaction strength (best first):
   1. (age, length_of_stay)
   2. (priors_count, length_of_stay)
   3. (length_of_stay, sex)
   4. (length_of_stay, race)
   5. (length_of_stay, c_charge_degree)
   6. (age, priors_count)
   7. (age, sex)
   8. (age, race)
   9. (age, c_charge_degree)
  10. (priors_count, race)
  11. (priors_count, sex)
  12. (race, sex)
  13. (priors_count, c_charge_degree)
  14. (c_charge_degree, race)
  15. (c_charge_degree, sex)

Top-10 selected for block training:
  (age, length_of_stay)
  (priors_count, length_of_stay)
  (length_of_stay, sex)
  (length_of_stay, race)
  (length_of_stay, c_charge_degree)
  (age, priors_count)
  (age, sex)
  (age, race)
  (age, c_charge_degree)
  (priors_count, race)
